In [2]:
import zipfile

zip_path = "/Users/aishanipradhan/Desktop/capstone/HeatmapData.zip"
extract_to = "/Users/aishanipradhan/Desktop/capstone/unzipped_contents"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_to)

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Events

In [4]:
events_1 = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer1-events_1-1-2025-to-3-31-2025.csv')
events_2 = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer2-events_1-1-2025-to-3-31-2025.csv')

In [5]:
def overview(df):
    print("DataFrame Overview:")
    print("---------------------")
    print(f"Number of rows: {df.shape[0]}")
    print("\nColumn Data Types:")
    print(df.dtypes)
    print("\nMissing Values per Column:")
    print(df.isnull().sum())
    print("\nUnique CUSTOMER_IDs:")
    print(df["CUSTOMER_ID"].unique())
    print("\nEVENT_TYPE Counts:")
    print(df["EVENT_TYPE"].value_counts())
    print("\nEVENT_SRC Counts:")
    print(df["EVENT_SRC"].value_counts())
    print("\nMETA_MODE Counts:")
    print(df["META_MODE"].value_counts())
    print("\nMETA_ENCODED_PROTO Counts:")
    print(df["META_ENCODED_PROTO"].value_counts())
    print("\nSESSION_ID and EVENT_TYPE Grouped Counts:")
    print(df.groupby(["SESSION_ID", "EVENT_TYPE"])["LOGGED_AT"].count())
    print("\nTimestamp Range:")
    print(f"Earliest timestamp: {df['LOGGED_AT'].min()}")
    print(f"Latest timestamp: {df['LOGGED_AT'].max()}")
    

In [6]:
events = pd.concat([events_1, events_2], ignore_index=True)

In [7]:
# customer 1 events overview
overview(events)

DataFrame Overview:
---------------------
Number of rows: 111440

Column Data Types:
EVENT_ID                       object
EVENT_AT                       object
EVENT_TYPE                     object
SERIAL_ID                      object
SESSION_ID                     object
CUSTOMER_ID                    object
INSTRUMENT_ID                  object
CONFIGURATION_ID               object
WORKER_ID                      object
COMPANY_ID                     object
DEPARTMENT_ID                  object
SENSOR_LIST                    object
EVENT_SRC                      object
LOGGED_AT                      object
LOGGED_AT_DATE                 object
RECEIVED_AT                    object
CREATED_AT                     object
INSTRUMENT_EVENT_ID             int64
LOGGED_VS_RECEIVED              int64
LOGGED_VS_CREATED               int64
RECEIVED_VS_CREATED             int64
META                           object
META_RECORD_NAME              float64
META_ENCODED_DATA             float64
MET

### Notes:
1. WORKER_ID, all meta variables other than meta mode and meta encoded proto mostly null - check if issue with how data was extracted.

2. Both customers - no discernable events, it enters and exits in normal mode throughout. No alarms, warnings, etc. But EVENT_SRC is instrument across the data - should there be battery/location etc present in EVENT_TYPE?

3. For the same session id, number of event normal mode doesnt match number of exit normal mode

4. Values in meta mode - calibrating, normal, off, charging.

5. META_ENCODED_PROTO decryption? Values - CAM=, CAI=, CAE=, CAQ=

6. One session_id can have multiple event_ids, but each session_id has one serial_id, instrument_id and configuration id

# Readings

In [8]:
readings_1 = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer1-readings_1-1-2025-to-3-31-2025.csv')
readings_2 = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer2-readings_1-1-2025-to-3-31-2025.csv')

/var/folders/_5/zwklnk315yqbrfhydp3_pj5w0000gn/T/ipykernel_3646/2290195512.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  readings_1 = pd.read_csv('/Users/aishanipradhan/Desktop/capstone/unzipped_contents/Data/customer1-readings_1-1-2025-to-3-31-2025.csv')


In [9]:
readings = pd.concat([readings_1, readings_2], ignore_index=True)

In [10]:
readings.head()

,GRID_SOURCE,OP,READING_ID,CUSTOMER_ID,SESSION_ID,READ_VALUE,PEAK_VALUE,MIN_VALUE,READING_SOURCE,INSTRUMENT_SERIAL,RECORDING_TIMESTAMP,RECORDING_DATE,GAS_TYPE,SENSOR_SITE,CREATED_AT,UNITS,__LOADED_AT,__TRANSACTION_AT,IS_DELETED
0,grid_prod_data,NaN,9025e4ef-e838-47a7-8d8c-64b994588ac4,1638b8ad-4e9d-4120-bbc2-27041d1ee1e5,2dea3ea2-57fe-416a-b712-97db37c7d923,0.0,NaN,NaN,LIVE,001E001001630305,2025-03-03 23:35:51.000,2025-03-03,SC_GASTYPE_DUOTOX_H2S,-1,2025-03-03 23:35:54.080,UNIT_PPM,2025-04-29 06:34:53.000,2025-04-29 06:34:08.910,False
1,grid_prod_data,NaN,587d2a43-69f7-411f-a04a-8072f2818b93,1638b8ad-4e9d-4120-bbc2-27041d1ee1e5,2dea3ea2-57fe-416a-b712-97db37c7d923,0.0,NaN,NaN,LIVE,001E001001630305,2025-03-03 21:45:51.000,2025-03-03,SC_GASTYPE_DUOTOX_H2S,-1,2025-03-03 21:45:54.306,UNIT_PPM,2025-04-29 06:33:57.000,2025-04-29 06:33:52.057,False
2,grid_prod_data,NaN,a5d5f660-3e8b-48e0-9092-3c2efc6fc0b2,1638b8ad-4e9d-4120-bbc2-27041d1ee1e5,2dea3ea2-57fe-416a-b712-97db37c7d923,0.0,NaN,NaN,LIVE,001E001001630305,2025-03-04 02:25:51.000,2025-03-04,SC_GASTYPE_DUOTOX_CO,-1,2025-03-04 02:25:53.993,UNIT_PPM,2025-04-29 06:34:53.000,2025-04-29 06:34:17.145,False
3,grid_prod_data,NaN,85c79268-7787-4c45-a710-4e5bed1b509b,1638b8ad-4e9d-4120-bbc2-27041d1ee1e5,2dea3ea2-57fe-416a-b712-97db37c7d923,0.0,NaN,NaN,LIVE,001E001001630305,2025-03-03 19:00:51.000,2025-03-03,SC_GASTYPE_DUOTOX_CO,-1,2025-03-03 19:00:54.885,UNIT_PPM,2025-04-29 06:33:57.000,2025-04-29 06:33:29.284,False
4,grid_prod_data,NaN,fe4532d3-666f-4cc2-9be1-f763ee2e8668,1638b8ad-4e9d-4120-bbc2-27041d1ee1e5,2dea3ea2-57fe-416a-b712-97db37c7d923,0.0,NaN,NaN,LIVE,001E001001630305,2025-03-04 01:30:51.000,2025-03-04,SC_GASTYPE_DUOTOX_H2S,-1,2025-03-04 01:30:53.762,UNIT_PPM,2025-04-29 06:34:53.000,2025-04-29 06:34:16.030,False


In [11]:
def overview_readings(df):
    print("DataFrame Overview:")
    print("---------------------")
    print(f"Number of rows: {df.shape[0]}")
    print("\nColumn Data Types:")
    print(df.dtypes)
    print("\nMissing Values per Column:")
    print(df.isnull().sum())
    print("\nUnique CUSTOMER_IDs:")
    print(df["CUSTOMER_ID"].unique())
    print("\nREAD_VALUE Counts:")
    print(df["READ_VALUE"].value_counts())
    print("\nSENSOR_SITE Counts:")
    print(df["SENSOR_SITE"].value_counts())
    print("\nTimestamp Range:")
    print(f"Earliest timestamp: {df['RECORDING_TIMESTAMP'].min()}")
    print(f"Latest timestamp: {df['RECORDING_TIMESTAMP'].max()}")

In [12]:
overview_readings(readings)

DataFrame Overview:
---------------------
Number of rows: 21280212

Column Data Types:
GRID_SOURCE             object
OP                      object
READING_ID              object
CUSTOMER_ID             object
SESSION_ID              object
READ_VALUE             float64
PEAK_VALUE             float64
MIN_VALUE              float64
READING_SOURCE          object
INSTRUMENT_SERIAL       object
RECORDING_TIMESTAMP     object
RECORDING_DATE          object
GAS_TYPE                object
SENSOR_SITE              int64
CREATED_AT              object
UNITS                   object
__LOADED_AT             object
__TRANSACTION_AT        object
IS_DELETED                bool
dtype: object

Missing Values per Column:
GRID_SOURCE                   0
OP                     21279412
READING_ID                    0
CUSTOMER_ID                   0
SESSION_ID                    0
READ_VALUE                    0
PEAK_VALUE             21280212
MIN_VALUE              21280212
READING_SOURCE            

## Notes
1. Peak value and minimum value missing from full data, sensor site consistently -1 - because only recorded for legacy devices
2. Most of the data has read value = 0, can we assume this means no events occurred?

## Combined

In [ ]:
combined = pd.merge(events_1, readings_1, on=["CUSTOMER_ID", "SESSION_ID"], how="inner", suffixes=('_event', '_reading'))

In [15]:
events_not_in_readings = (
    events.loc[~events["SESSION_ID"].isin(readings["SESSION_ID"]), "SESSION_ID"]
    .unique()
)

print("Events SESSION_IDs not in Readings SESSION_IDs:")
print(events_not_in_readings)

Events SESSION_IDs not in Readings SESSION_IDs:
['ad74a159-55e4-4458-9612-76197a3551ec'
 'ad79bba2-0bc5-41e5-8ea1-7b0bd4d602aa'
 'bd966f95-ef53-49bf-b5de-b96924333ba4' ...
 '9a561625-970d-4c28-8553-da238e273d7b'
 '9a5736ef-ae29-4b36-b0fc-a73b04f1b6c7'
 '9a5c2926-3705-49bc-af26-40a264a80ed9']


In [16]:
readings_not_in_events = (
    readings.loc[~readings["SESSION_ID"].isin(events["SESSION_ID"]), "SESSION_ID"]
    .unique()
)
print("Readings SESSION_IDs not in Events SESSION_IDs:")
print(readings_not_in_events)

Readings SESSION_IDs not in Events SESSION_IDs:
['3b38a3fd-1f54-40d6-8395-1bdc8eff811f'
 'dd07f7b7-5b1b-4b78-b098-95ed9bdb1e2e']


In [ ]:
combined = pd.merge(events, readings, on=["CUSTOMER_ID", "SESSION_ID"], how="inner", suffixes=('_event', '_reading'))

## Notes
1. What could be cause of certain sessions present in events but not in readings and vice versa?
2. Kernel crashing every time we try to join events and readings (tried inner, outer, left)

MAJOR ISSUES:
1. No location data
2. No event data in terms of the types of events (alarm/notification/location)